<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/05_week/model_calibration_and_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F

# TR: Modelin ham çıktılarını (logit) Softmax fonksiyonu ile olasılık dağılımına dönüştürür.
# EN: Converts raw model outputs (logits) into a probability distribution using Softmax.
def code_01():
    logits = torch.tensor([2.0, 1.0, 0.1])
    probs = F.softmax(logits, dim=-1)
    return probs

print("Kod 1 Çıktısı:", code_01())

Kod 1 Çıktısı: tensor([0.6590, 0.2424, 0.0986])


In [ ]:
import torch

# TR: Olasılık dağılımı içerisinden en yüksek değerli tahmini ve modelin güven skorunu (confidence) bulur.
# EN: Finds the top prediction and its corresponding confidence score from a probability distribution.
def code_02():
    probs = torch.tensor([0.70, 0.20, 0.10])
    conf, pred = torch.max(probs, dim=-1)
    return conf.item(), pred.item()

print("Kod 2 Çıktısı (Güven, Tahmin Class):", code_02())

Kod 2 Çıktısı (Güven, Tahmin Class): (0.699999988079071, 0)


In [ ]:
import torch

# TR: Bir olasılık dağılımının bilgi teorisindeki Shannon Entropisini (genel belirsizlik) hesaplar.
# EN: Calculates the Shannon Entropy (total uncertainty) of a probability distribution in information theory.
def code_03():
    probs = torch.tensor([0.33, 0.33, 0.34])
    entropy = -torch.sum(probs * torch.log(probs + 1e-12))
    return entropy.item()

print("Kod 3 Çıktısı (Entropi):", code_03())

Kod 3 Çıktısı (Entropi): 1.0985126495361328


In [ ]:
import torch

# TR: Sigmoid aktivasyonu sonrası ikili sınıflandırmada modelin karar güvenini belirler.
# EN: Determines the decision confidence of a binary classification model after sigmoid activation.
def code_04():
    logit = torch.tensor([1.5])
    prob = torch.sigmoid(logit)
    conf = torch.max(prob, 1 - prob)
    return conf.item()

print("Kod 4 Çıktısı:", code_04())

Kod 4 Çıktısı: 0.8175744414329529


In [ ]:
import torch

# TR: Model tahminlerinin doğruluğu ile güven değerlerini karşılaştırma için eşleştirir.
# EN: Matches prediction accuracy with confidence values for comparison purposes.
def code_05():
    preds = torch.tensor([0, 1, 2])
    targets = torch.tensor([0, 1, 1])
    confs = torch.tensor([0.9, 0.8, 0.6])
    accs = (preds == targets).float()
    return accs, confs

print("Kod 5 Çıktısı (Doğruluklar, Güvenler):", code_05())

Kod 5 Çıktısı (Doğruluklar, Güvenler): (tensor([1., 1., 0.]), tensor([0.9000, 0.8000, 0.6000]))


In [ ]:
import torch

# TR: Güven skorlarını ECE (Expected Calibration Error) hesabı için belirli aralıklara (bins) böler.
# EN: Bins confidence scores into specific intervals for ECE (Expected Calibration Error) calculation.
def code_06():
    confs = torch.tensor([0.15, 0.35, 0.65, 0.85])
    n_bins = 5
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    bin_indices = torch.bucketize(confs, bin_boundaries)
    return bin_indices

print("Kod 6 Çıktısı (Bin İndeksleri):", code_06())

Kod 6 Çıktısı (Bin İndeksleri): tensor([1, 2, 4, 5])


In [ ]:
import torch

# TR: Tek bir kovana (bin) düşen tahminlerin ortalama güven değerini hesaplar.
# EN: Calculates the average confidence value of predictions belonging to a single bin.
def code_07():
    confs = torch.tensor([0.82, 0.88, 0.85])
    avg_conf = torch.mean(confs)
    return avg_conf.item()

print("Kod 7 Çıktısı:", code_07())

Kod 7 Çıktısı: 0.8500000834465027


In [ ]:
import torch

# TR: Tek bir kovana (bin) düşen tahminlerin gerçek doğruluk (accuracy) oranını bulur.
# EN: Finds the actual accuracy rate of predictions within a single bin.
def code_08():
    accs = torch.tensor([1.0, 0.0, 1.0])
    avg_acc = torch.mean(accs)
    return avg_acc.item()

print("Kod 8 Çıktısı:", code_08())

Kod 8 Çıktısı: 0.6666666865348816


In [ ]:
import torch

# TR: Basit bir kovan yapısı üzerinden ilk seviye ECE skorunu hesaplar.
# EN: Computes a basic first-level ECE score over a simple binning structure.
def code_09():
    confs = torch.tensor([0.9, 0.8, 0.4, 0.2])
    accs = torch.tensor([1.0, 0.0, 0.0, 0.0])
    ece = torch.mean(torch.abs(confs - accs))
    return ece.item()

print("Kod 9 Çıktısı (Temel ECE):", code_09())

Kod 9 Çıktısı (Temel ECE): 0.3750000298023224


In [ ]:
import torch

# TR: Kovanlar arasındaki maksimum sapmayı (en kötü kalibrasyon hatasını) hesaplar.
# EN: Calculates the maximum calibration error (the worst bin gap across all bins).
def code_10():
    gaps = torch.tensor([0.05, 0.40, 0.12, 0.02])
    mce = torch.max(gaps)
    return mce.item()

print("Kod 10 Çıktısı (MCE):", code_10())

Kod 10 Çıktısı (MCE): 0.4000000059604645


In [11]:
import torch

# TR: Tahmin edilen olasılıkların gerçek etiketlerden ortalama karesel sapmasını ölçer.
# EN: Measures the mean squared difference between predicted probabilities and actual targets.
def code_11():
    probs = torch.tensor([0.9, 0.2, 0.7])
    targets = torch.tensor([1.0, 0.0, 0.0])
    brier = torch.mean((probs - targets) ** 2)
    return brier.item()

print("Kod 11 Çıktısı (Brier Score):", code_11())

Kod 11 Çıktısı (Brier Score): 0.17999999225139618


In [12]:
import torch
import torch.nn.functional as F

# TR: Çok sınıflı (multi-class) olasılık dağılımları için Brier skorunu hesaplar.
# EN: Calculates the Brier Score for multi-class probability distributions.
def code_12():
    probs = torch.tensor([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1]])
    targets = F.one_hot(torch.tensor([0, 2]), num_classes=3).float()
    brier = torch.mean(torch.sum((probs - targets) ** 2, dim=-1))
    return brier.item()

print("Kod 12 Çıktısı:", code_12())

Kod 12 Çıktısı: 0.800000011920929


In [13]:
import torch
import torch.nn.functional as F

# TR: Modelin kalibrasyon kalitesini ve belirsizliğini cezalandıran NLL olasılıksal kayıp fonksiyonu.
# EN: Probabilistic NLL loss function evaluating model calibration and uncertainty quality.
def code_13():
    logits = torch.tensor([[2.0, 1.0], [0.5, 2.5]])
    targets = torch.tensor([0, 1])
    nll = F.cross_entropy(logits, targets)
    return nll.item()

print("Kod 13 Çıktısı (NLL):", code_13())

Kod 13 Çıktısı (NLL): 0.22009485960006714


In [15]:
import torch

# TR: Verilen tüm güven ve doğruluk dizileri için kovanlı tam ECE metriğini hesaplayan fonksiyon.
# EN: Comprehensive function calculating binned ECE metrics for given confidence and accuracy arrays.
def code_14(confs, accs, n_bins=10):
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    ece = torch.tensor(0.0)
    for i in range(n_bins):
        in_bin = (confs > bin_boundaries[i]) & (confs <= bin_boundaries[i+1])
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            accuracy_in_bin = accs[in_bin].mean()
            avg_confidence_in_bin = confs[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece.item()

confs = torch.tensor([0.9, 0.8, 0.7, 0.4, 0.2])
accs = torch.tensor([1.0, 1.0, 0.0, 0.0, 0.0])
print("Kod 14 Çıktısı (Detaylı ECE):", code_14(confs, accs))

Kod 14 Çıktısı (Detaylı ECE): 0.3199999928474426


In [16]:
import torch

# TR: Modelin ortalama güveninin gerçek ortalama doğruluktan ne kadar yüksek sapma gösterdiğini ölçer.
# EN: Measures how much the model's average confidence exceeds its actual average accuracy.
def code_15():
    confs = torch.tensor([0.95, 0.90, 0.85, 0.99])
    accs = torch.tensor([1.0, 0.0, 1.0, 0.0])
    overconfidence_gap = torch.mean(confs) - torch.mean(accs)
    return overconfidence_gap.item()

print("Kod 15 Çıktısı (Overconfidence Gap):", code_15())

Kod 15 Çıktısı (Overconfidence Gap): 0.42249995470046997


In [17]:
import torch
import torch.nn.functional as F

# TR: Logit değerlerini bir sıcaklık katsayısına (T) bölerek olasılık dağılımını yumuşatır/keskinleştirir.
# EN: Scales logit values using a temperature parameter (T) to smooth or sharpen probability distributions.
def code_16(logits, T=1.5):
    scaled_logits = logits / T
    return F.softmax(scaled_logits, dim=-1)

logits = torch.tensor([3.0, 1.0, 0.2])
print("Kod 16 Çıktısı (Ölçeklenmiş Probs):", code_16(logits))

Kod 16 Çıktısı (Ölçeklenmiş Probs): tensor([0.7051, 0.1859, 0.1090])


In [18]:
import torch
import torch.nn.functional as F

# TR: Düşük ve yüksek sıcaklık değerlerinin olasılık entropisi üzerindeki değişimini kıyaslar.
# EN: Compares the impact of low vs. high temperature values on probability entropy.
def code_17():
    logits = torch.tensor([3.0, 1.0, 0.2])
    p_low = F.softmax(logits / 0.5, dim=-1)
    p_high = F.softmax(logits / 2.0, dim=-1)
    e_low = -torch.sum(p_low * torch.log(p_low + 1e-12))
    e_high = -torch.sum(p_high * torch.log(p_high + 1e-12))
    return e_low.item(), e_high.item()

print("Kod 17 Çıktısı (Düşük T Entropisi, Yüksek T Entropisi):", code_17())

Kod 17 Çıktısı (Düşük T Entropisi, Yüksek T Entropisi): (0.11372127383947372, 0.92071133852005)


In [19]:
import torch
import torch.nn.functional as F

# TR: Optimal T (Sıcaklık) parametresini bulmak için kullanılacak Cross-Entropy kaybını hazırlar.
# EN: Prepares the Cross-Entropy loss used to optimize the optimal Temperature (T) parameter.
def code_18(logits, targets, T_param):
    scaled_logits = logits / T_param
    return F.cross_entropy(scaled_logits, targets)

logits = torch.randn(5, 3)
targets = torch.tensor([0, 1, 2, 0, 1])
T_param = torch.nn.Parameter(torch.tensor([1.5]))
print("Kod 18 Çıktısı (NLL Kaybı):", code_18(logits, targets, T_param).item())

Kod 18 Çıktısı (NLL Kaybı): 0.7402222752571106


In [21]:
import torch
import torch.nn.functional as F
import numpy as np

# TR: Doğrulama kümesi üzerinde NLL kaybını en küçükleyen en iyi T sıcaklık değerini arar.
# EN: Searches for the optimal temperature T that minimizes NLL loss on validation set.
def code_19_search():
    logits = torch.randn(100, 5)
    targets = torch.randint(0, 5, (100,))
    best_T, min_nll = 1.0, float('inf')
    for T in np.linspace(0.1, 3.0, 30):
        loss = F.cross_entropy(logits / T, targets).item()
        if loss < min_nll:
            min_nll, best_T = loss, T
    return best_T

print("Kod 19 Çıktısı (En İyi T):", code_19_search())

Kod 19 Çıktısı (En İyi T): 3.0


In [20]:
import torch

# TR: Belirlenen bir güven eşiğinin altında kalan hatalı olabilecek tahminleri eler/filtreler.
# EN: Filters out low-confidence predictions falling below a predefined confidence threshold.
def code_20(probs, threshold=0.8):
    confs, preds = torch.max(probs, dim=-1)
    mask = confs >= threshold
    return preds[mask], confs[mask]

probs = torch.tensor([[0.9, 0.1], [0.55, 0.45], [0.85, 0.15]])
preds, confs = code_20(probs)
print("Kod 20 Çıktısı (Filtrelenmiş Tahminler):", preds)

Kod 20 Çıktısı (Filtrelenmiş Tahminler): tensor([0, 0])
